# Final model comparison

This notebook consolidates final temporal-test performance. It does not retrain models or select features.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
import yaml
import joblib
import pandas as pd
from IPython.display import display

CONFIG_PATH = Path('../../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.final_comparison import (
    evaluate_models_on_table, add_feature_metadata, rank_models, save_final_comparison,
)

In [4]:
prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.parquet'
if not prediction_path.exists():
    prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.csv'
predictions = load_admission_file(prediction_path)
target = config['columns']['target']
target_mapping = config.get('target_mapping', config.get('analysis', {}).get('target_mapping', {}))
test = predictions[predictions['split'].eq('test')].copy()
prediction_columns = [c for c in test.columns if c.endswith('_prob')]
print(f'Test admissions: {len(test):,}')
print(f'Model prediction columns: {len(prediction_columns)}')

Test admissions: 2,483
Model prediction columns: 6


In [5]:
comparison = evaluate_models_on_table(
    test, target_column=target, prediction_columns=prediction_columns,
    target_mapping=target_mapping,
    threshold=config.get('analysis', {}).get('evaluation_threshold', 0.5),
)
model_configs = config.get('models', [])
feature_counts = {}
for model_cfg in model_configs:
    path = Path(config['outputs']['model_directory']) / f"{model_cfg['name']}.joblib"
    if path.exists():
        bundle = joblib.load(path)
        if isinstance(bundle, dict) and bundle.get('features') is not None:
            feature_counts[model_cfg['name']] = len(bundle['features'])
comparison = add_feature_metadata(comparison, model_configs=model_configs, feature_counts=feature_counts)
ranked = rank_models(comparison)
display(ranked)

,model,prediction_column,n,n_events,event_rate,auroc,auprc,brier_score,threshold,tp,...,balanced_accuracy,f1,calibration_intercept,calibration_slope,observed_expected_ratio,feature_set,model_type,n_features,primary_rank,composite_score
0,xgb_kim_components,xgb_kim_components_prob,2483,507,0.204188,0.504129,0.205492,0.162516,0.5,0,...,0.500000,0.000000,-0.458252,0.643512,1.034100,kim_components,xgboost,5,5.0,2.666667
1,lr_kim_components,lr_kim_components_prob,2483,507,0.204188,0.504129,0.205492,0.249826,0.5,453,...,0.503932,0.334317,-1.360077,0.658956,0.408652,kim_components,logistic_regression,5,5.0,2.166667
2,xgb_all_score_values,xgb_all_score_values_prob,2483,507,0.204188,0.510003,0.209188,0.162587,0.5,0,...,0.500000,0.000000,-0.881526,0.341757,1.031020,all_score_values,xgboost,6,3.0,1.666667
3,lr_all_score_values,lr_all_score_values_prob,2483,507,0.204188,0.510027,0.209112,0.249847,0.5,453,...,0.505450,0.335059,-1.360328,0.586501,0.408679,all_score_values,logistic_regression,6,2.0,1.166667
4,lr_clinical_baseline,lr_clinical_baseline_prob,2483,507,0.204188,0.507487,0.211771,0.255676,0.5,273,...,0.498988,0.295455,-1.364527,0.152477,0.403400,clinical_baseline,logistic_regression,16,4.0,1.000000
5,xgb_clinical_baseline,xgb_clinical_baseline_prob,2483,507,0.204188,0.516103,0.216840,0.163934,0.5,0,...,0.499747,0.000000,-1.041989,0.229540,1.004360,clinical_baseline,xgboost,16,1.0,0.833333


In [6]:
output_dir = Path(config['outputs'].get('final_directory', 'results/final'))
output_dir.mkdir(parents=True, exist_ok=True)
ranked.to_csv(output_dir / 'final_model_comparison.csv', index=False)
ranked.to_csv(output_dir / 'final_model_ranking.csv', index=False)
print(f'Saved results to {output_dir.resolve()}')

Saved results to /app/notebooks/c_analysis/results/final
